In [ ]:
!pip install pyspark

1. Basic MapReduce Implementation

In [ ]:
from functools import reduce
from collections import defaultdict

class SimpleMapReduce:
    def __init__(self, mapper, reducer):
        self.mapper = mapper
        self.reducer = reducer

    def __call__(self, data):
        # Map phase
        mapped_data = []
        for item in data:
            mapped_data.extend(self.mapper(item))

        # Shuffle phase (group by key)
        grouped_data = defaultdict(list)
        for key, value in mapped_data:
            grouped_data[key].append(value)

        # Reduce phase
        return [self.reducer(key, values)
                for key, values in grouped_data.items()]

# Example: Word Count
def word_count_mapper(document):
    words = document.lower().split()
    return [(word, 1) for word in words]

def word_count_reducer(word, counts):
    return (word, sum(counts))

# Test
documents = [
    "hello world python map reduce",
    "hello python world",
    "map reduce is powerful"
]

mapreduce = SimpleMapReduce(word_count_mapper, word_count_reducer)
result = mapreduce(documents)
print("Word Count:", dict(result))

Word Count: {'hello': 2, 'world': 2, 'python': 2, 'map': 2, 'reduce': 2, 'is': 1, 'powerful': 1}


2. Word Count with Built-in Function


In [ ]:
from collections import Counter
from functools import reduce

def map_function(document):
    """Map: Split document into word-count pairs"""
    return [(word, 1) for word in document.lower().split()]

def reduce_function(word_counts):
    """Reduce: Sum counts for each word"""
    return reduce(lambda x, y: (x[0], x[1] + y[1]), word_counts)

# Using Python's built-in functions
documents = [
    "the quick brown fox jumps over the lazy dog",
    "the fox is quick and the dog is lazy",
    "brown fox brown dog"
]

# Map phase
mapped = [pair for doc in documents for pair in map_function(doc)]

# Shuffle phase
shuffled = {}
for word, count in mapped:
    shuffled.setdefault(word, []).append((word, count))

# Reduce phase
reduced = {word: reduce_function(values)[1]
           for word, values in shuffled.items()}

print("Word Count:", reduced)

Word Count: {'the': 4, 'quick': 2, 'brown': 3, 'fox': 3, 'jumps': 1, 'over': 1, 'lazy': 2, 'dog': 3, 'is': 2, 'and': 1}


Foundation for search engines, NLP, content analysis

Used in spam detection, sentiment analysis, keyword extraction

Google's original MapReduce paper used this as the primary example

3. Multi-step MapReduce: Average Calculation

In [ ]:
from collections import defaultdict
import math

def map_average(data):
    """Map: Emit (key, (value, 1)) for partial sums"""
    return [(item['category'], (item['value'], 1)) for item in data]

def reduce_average(key, values):
    """Reduce: Calculate sum and count, then average"""
    total_sum = sum(v[0] for v in values)
    total_count = sum(v[1] for v in values)
    return (key, total_sum / total_count)

# Example data
data = [
    {'category': 'A', 'value': 10},
    {'category': 'B', 'value': 20},
    {'category': 'A', 'value': 30},
    {'category': 'B', 'value': 40},
    {'category': 'A', 'value': 50},
    {'category': 'C', 'value': 60}
]

# Map
mapped = map_average(data)

# Shuffle
shuffled = defaultdict(list)
for key, value in mapped:
    shuffled[key].append(value)

# Reduce
averages = [reduce_average(key, values) for key, values in shuffled.items()]
print("Averages:", dict(averages))

Averages: {'A': 30.0, 'B': 30.0, 'C': 60.0}


Powers every search engine (Google, Elasticsearch, Solr)

E-commerce product search

Document management systems

Real-world: Any "search box" feature

**Spark**

In [ ]:
#Sales Analysis
from pyspark.sql.functions import col, sum, avg
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# Create a Spark session
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("SalesAnalysis").getOrCreate()

# Sample sales data
data = [
    (2023, "Product A", 100, 10),
    (2023, "Product B", 50, 15),
    (2023, "Product C", 75, 12),
    (2024, "Product A", 120, 12),
    (2024, "Product B", 80, 18),
    (2024, "Product C", 90, 15)
]

# Create a DataFrame from the sample data
schema = StructType([
    StructField("year", IntegerType(), True),
    StructField("product", StringType(), True),
    StructField("units_sold", IntegerType(), True),
    StructField("price", IntegerType(), True)
])
df = spark.createDataFrame(data, schema)

# Sales analysis
sales_summary = (
    df
    .groupBy("year", "product")
    .agg(
        sum("units_sold").alias("total_units_sold"),
        sum(col("units_sold") * col("price")).alias("total_revenue"),
        avg("price").alias("average_price")
    )
    .orderBy("year", "product")
)

# Display the results
sales_summary.show(truncate=False)

+----+---------+----------------+-------------+-------------+
|year|product  |total_units_sold|total_revenue|average_price|
+----+---------+----------------+-------------+-------------+
|2023|Product A|100             |1000         |10.0         |
|2023|Product B|50              |750          |15.0         |
|2023|Product C|75              |900          |12.0         |
|2024|Product A|120             |1440         |12.0         |
|2024|Product B|80              |1440         |18.0         |
|2024|Product C|90              |1350         |15.0         |
+----+---------+----------------+-------------+-------------+



In [ ]:
#Customer Segmentation
from pyspark.sql.functions import col, length, when
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# Create a Spark session
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("CustomerSegmentation").getOrCreate()

# Sample customer data
data = [
    (1, "John Doe", "premium", 1000),
    (2, "Jane Smith", "regular", 500),
    (3, "Bob Johnson", "premium", 2000),
    (4, "Sarah Lee", "basic", 300),
    (5, "Mike Brown", "regular", 800)
]

# Create a DataFrame from the sample data
schema = StructType([
    StructField("customer_id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("membership_type", StringType(), True),
    StructField("spend_amount", IntegerType(), True)
])
df = spark.createDataFrame(data, schema)

# Customer segmentation
customer_segments = (
    df
    .withColumn("name_length", length("name"))
    .withColumn("segment", when(col("spend_amount") >= 1000, "high_value")
                          .when(col("spend_amount") >= 500, "medium_value")
                          .otherwise("low_value"))
    .orderBy("segment", "spend_amount", "name_length")
)

# Display the results
customer_segments.show(truncate=False)

+-----------+-----------+---------------+------------+-----------+------------+
|customer_id|name       |membership_type|spend_amount|name_length|segment     |
+-----------+-----------+---------------+------------+-----------+------------+
|1          |John Doe   |premium        |1000        |8          |high_value  |
|3          |Bob Johnson|premium        |2000        |11         |high_value  |
|4          |Sarah Lee  |basic          |300         |9          |low_value   |
|2          |Jane Smith |regular        |500         |10         |medium_value|
|5          |Mike Brown |regular        |800         |10         |medium_value|
+-----------+-----------+---------------+------------+-----------+------------+



In [ ]:
import pandas as pd
import numpy as np
import time
import tempfile
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, avg, count, window, desc, when, rand

# Initialize Spark Session
spark = SparkSession.builder \
    .appName("ImportanceOfSpark") \
    .config("spark.sql.shuffle.partitions", "4") \
    .master("local[*]") \
    .getOrCreate()

print("="*70)
print("WHY APACHE SPARK IS IMPORTANT - WORKING EXAMPLES")
print("="*70)

# ============================================
# DEMO 1: Memory Management (Pandas vs Spark)
# ============================================
print("\n📊 DEMO 1: MEMORY MANAGEMENT")
print("-"*50)

# Create a dataset that would be large in memory
n_rows = 5_000_000
print(f"Creating {n_rows:,} rows of data...")

# Pandas approach - loads everything into memory
print("\n【PANDAS】Loading data into memory...")
start_time = time.time()
pandas_df = pd.DataFrame({
    'id': range(n_rows),
    'value': np.random.randn(n_rows),
    'category': np.random.choice(['A', 'B', 'C', 'D'], n_rows),
    'timestamp': pd.date_range('2024-01-01', periods=n_rows, freq='S')
})
pandas_time = time.time() - start_time
pandas_memory = pandas_df.memory_usage(deep=True).sum() / 1024**2

print(f"  ✓ Loaded in {pandas_time:.2f} seconds")
print(f"  💾 Memory usage: {pandas_memory:.0f} MB")
print(f"  ⚠️  Entire dataset in RAM - limits size to available memory")

# Spark approach - lazy loading
print("\n【SPARK】Creating distributed dataset...")
start_time = time.time()
spark_df = spark.range(n_rows)
spark_df = spark_df.withColumn('value', (spark_df.id - n_rows/2) / n_rows * 10)
spark_df = spark_df.withColumn('category',
    when(col('id') % 4 == 0, 'A')
    .when(col('id') % 4 == 1, 'B')
    .when(col('id') % 4 == 2, 'C')
    .otherwise('D'))
spark_time = time.time() - start_time

print(f"  ✓ Created in {spark_time:.2f} seconds (lazy evaluation)")
print(f"  💾 No memory used until action performed")
print(f"  ✅ Can handle datasets much larger than RAM")

# ============================================
# DEMO 2: Complex Aggregation Performance
# ============================================
print("\n📊 DEMO 2: COMPLEX AGGREGATION PERFORMANCE")
print("-"*50)

# Pandas aggregation
print("\n【PANDAS】Complex aggregation...")
start_time = time.time()
pandas_result = pandas_df.groupby('category').agg({
    'value': ['mean', 'std', 'count'],
    'id': 'count'
})
pandas_agg_time = time.time() - start_time
print(f"  ✓ Completed in {pandas_agg_time:.2f} seconds")
print(f"  Results:")
print(pandas_result.head())

# Spark aggregation (parallel processing)
print("\n【SPARK】Complex aggregation (parallel)...")
start_time = time.time()
spark_result = spark_df.groupBy('category').agg(
    avg('value').alias('mean_value'),
    count('value').alias('count'),
    sum('value').alias('sum_value')
)
# Trigger action
spark_result.show(5, truncate=False)
spark_agg_time = time.time() - start_time
print(f"  ✓ Completed in {spark_agg_time:.2f} seconds")
print(f"  🚀 Speed advantage: {pandas_agg_time/spark_agg_time:.1f}x faster")

# ============================================
# DEMO 3: Handling Data Larger Than RAM
# ============================================
print("\n📊 DEMO 3: DATA LARGER THAN RAM SIMULATION")
print("-"*50)

# Create multiple CSV files simulating big data
temp_dir = tempfile.mkdtemp()
num_files = 10
rows_per_file = 500_000

print(f"Creating {num_files} CSV files with {rows_per_file:,} rows each...")
for i in range(num_files):
    df_temp = pd.DataFrame({
        'user_id': np.random.randint(1, 10000, rows_per_file),
        'amount': np.random.uniform(10, 1000, rows_per_file),
        'region': np.random.choice(['North', 'South', 'East', 'West'], rows_per_file)
    })
    df_temp.to_csv(f"{temp_dir}/data_{i}.csv", index=False)

total_rows = num_files * rows_per_file
total_size = __builtins__.sum(os.path.getsize(f"{temp_dir}/data_{i}.csv") for i in range(num_files)) / 1024**2
print(f"  Total data: {total_rows:,} rows, {total_size:.0f} MB")

# Pandas trying to load all files
print("\n【PANDAS】Attempting to load all files...")
try:
    start_time = time.time()
    all_data = pd.concat([pd.read_csv(f"{temp_dir}/data_{i}.csv")
                          for i in range(num_files)])
    pandas_load_time = time.time() - start_time
    print(f"  ✓ Loaded in {pandas_load_time:.2f} seconds")
    print(f"  ⚠️ Memory: {all_data.memory_usage(deep=True).sum() / 1024**2:.0f} MB")
except MemoryError:
    print("  ✗ MEMORY ERROR: Cannot load all files at once!")
except Exception as e:
    print(f"  ✗ Error: {e}")

# Spark reading all files in parallel
print("\n【SPARK】Reading all files in parallel...")
start_time = time.time()
spark_big_data = spark.read.csv(f"{temp_dir}/*.csv", header=True, inferSchema=True)
# Spark hasn't loaded data yet - just created execution plan
spark_prepare_time = time.time() - start_time

# Now perform computation
start_time = time.time()
result = spark_big_data.groupBy('region').agg(
    avg('amount').alias('avg_amount'),
    count('user_id').alias('user_count')
).collect()
spark_process_time = time.time() - start_time

print(f"  ✓ Prepared in {spark_prepare_time:.2f} seconds (lazy)")
print(f"  ✓ Computed in {spark_process_time:.2f} seconds")
print(f"  Results: {result}")
print(f"  ✅ Successfully processed data larger than memory!")

# ============================================
# DEMO 4: Fault Tolerance
# ============================================
print("\n📊 DEMO 4: FAULT TOLERANCE DEMONSTRATION")
print("-"*50)

# Create RDD that will simulate failure
rdd = spark.sparkContext.parallelize(range(1000), numSlices=10)

def process_with_potential_failure(x):
    """Simulate occasional failure"""
    if x == 500:  # Simulate one failing task
        raise Exception("Simulated task failure!")
    return x * 2

print("Processing with potential failures...")
try:
    # Spark automatically retries failed tasks
    result = rdd.map(process_with_potential_failure).collect()
    print(f"  ✓ Successfully processed all {len(result)} items despite failure")
    print(f"  🔄 Spark automatically retried the failed partition on another executor")
except Exception as e:
    print(f"  Error: {e}")

# ============================================
# DEMO 5: Caching for Iterative Algorithms
# ============================================
print("\n📊 DEMO 5: CACHING FOR ITERATIVE ALGORITHMS")
print("-"*50)

# Create dataset for iterative processing
iter_data = spark.range(1000000)
iter_data = iter_data.withColumn('value', rand())

# Without caching
print("\n【WITHOUT CACHING】Running same query multiple times...")
times = []
for i in range(3):
    start_time = time.time()
    count_result = iter_data.filter(col('value') > 0.5).count()
    elapsed = time.time() - start_time
    times.append(elapsed)
    print(f"  Run {i+1}: {elapsed:.3f} seconds (recomputed from source)")

print(f"  Average: {__builtins__.sum(times)/len(times):.3f} seconds")

# With caching
print("\n【WITH CACHING】Caching data in memory...")
iter_data.cache()
# Force cache
iter_data.count()

times_cached = []
for i in range(3):
    start_time = time.time()
    count_result = iter_data.filter(col('value') > 0.5).count()
    elapsed = time.time() - start_time
    times_cached.append(elapsed)
    print(f"  Run {i+1}: {elapsed:.3f} seconds (from cache)")

print(f"  Average: {__builtins__.sum(times_cached)/len(times_cached):.3f} seconds")
print(f"  🚀 Caching speedup: {__builtins__.sum(times)/__builtins__.sum(times_cached):.1f}x faster!")

# ============================================
# SUMMARY TABLE
# ============================================
print("\n" + "="*70)
print("KEY ADVANTAGES DEMONSTRATED")
print("="*70)

summary = """
┌─────────────────────────┬─────────────────────┬─────────────────────┐
│ FEATURE                 │ PANDAS              │ SPARK               │
├─────────────────────────┼─────────────────────┼─────────────────────┤
│ Memory limit            │ ~Available RAM      │ ✅ Scales to TB/PB  │
│ Processing mode         │ Single-threaded     │ ✅ Parallel across  │
│                         │                     │    all CPU cores    │
│ Fault tolerance         │ ❌ None             │ ✅ Automatic retry  │
│ Lazy evaluation         │ ❌ Eager            │ ✅ Optimizes plans  │
│ Caching for iteration   │ ❌ Manual           │ ✅ Built-in cache   │
│ Handles 10M+ rows       │ ⚠️  Slow/May crash  │ ✅ Fast & efficient │
│ Learning curve          │ ✅ Easy             │ ⚠️  Steeper         │
└─────────────────────────┴─────────────────────┴─────────────────────┘

REAL-WORLD USE CASES WHERE SPARK IS ESSENTIAL:
• Processing terabytes of log data
• Real-time stream processing
• Machine learning on large datasets
• ETL pipelines with complex transformations
• Interactive queries on big data
• Graph processing on social networks
"""

print(summary)

# Cleanup
spark.stop()
import shutil
shutil.rmtree(temp_dir)
print("\n✓ Cleanup complete - Spark session closed")

WHY APACHE SPARK IS IMPORTANT - WORKING EXAMPLES

📊 DEMO 1: MEMORY MANAGEMENT
--------------------------------------------------
Creating 5,000,000 rows of data...

【PANDAS】Loading data into memory...


/tmp/ipykernel_8358/3557234293.py:37: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  'timestamp': pd.date_range('2024-01-01', periods=n_rows, freq='S')


  ✓ Loaded in 2.53 seconds
  💾 Memory usage: 353 MB
  ⚠️  Entire dataset in RAM - limits size to available memory

【SPARK】Creating distributed dataset...
  ✓ Created in 0.22 seconds (lazy evaluation)
  💾 No memory used until action performed
  ✅ Can handle datasets much larger than RAM

📊 DEMO 2: COMPLEX AGGREGATION PERFORMANCE
--------------------------------------------------

【PANDAS】Complex aggregation...
  ✓ Completed in 0.71 seconds
  Results:
             value                          id
              mean       std    count    count
category                                      
A        -0.000083  0.999773  1250413  1250413
B        -0.000470  0.999375  1250492  1250492
C        -0.000520  1.000064  1249807  1249807
D        -0.002280  0.999463  1249288  1249288

【SPARK】Complex aggregation (parallel)...
+--------+----------------------+-------+----------------------+
|category|mean_value            |count  |sum_value             |
+--------+----------------------+-------+----

In [ ]:
"""
FAIR COMPARISON: Pandas vs Spark on Identical Data
Shows why Spark matters when data gets large
"""

import pandas as pd
import numpy as np
import time
import psutil
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, count, sum as spark_sum, stddev

# Initialize Spark
spark = SparkSession.builder \
    .appName("FairComparison") \
    .config("spark.sql.shuffle.partitions", "4") \
    .master("local[*]") \
    .getOrCreate()

print("="*80)
print("FAIR COMPARISON: PANDAS vs SPARK on IDENTICAL DATA")
print("="*80)

# ============================================
# STEP 1: Create ONE dataset for both tools
# ============================================
print("\n📦 STEP 1: Creating identical dataset...")

# Fixed random seed for reproducibility
np.random.seed(42)

# Dataset sizes to test (start small, then scale up)
test_sizes = [100_000, 500_000, 1_000_000, 5_000_000]

results = []

for size in test_sizes:
    print(f"\n{'='*80}")
    print(f"TESTING WITH {size:,} ROWS")
    print(f"{'='*80}")

    # Create the data (same for both)
    data = {
        'user_id': np.random.randint(1, 10000, size),
        'product_id': np.random.randint(1, 1000, size),
        'category': np.random.choice(['Electronics', 'Clothing', 'Books', 'Home'], size),
        'price': np.random.uniform(10, 500, size).round(2),
        'quantity': np.random.randint(1, 10, size),
        'region': np.random.choice(['North', 'South', 'East', 'West'], size)
    }

    # Create pandas DataFrame
    pandas_df = pd.DataFrame(data)

    # Create Spark DataFrame (same data)
    spark_df = spark.createDataFrame(pandas_df)

    # ============================================
    # TEST 1: Simple Aggregation (Group by + Average)
    # ============================================
    print(f"\n📊 TEST 1: Average price by category")
    print("-" * 50)

    # PANDAS
    print("Pandas: ", end="", flush=True)
    start_time = time.time()
    pandas_result1 = pandas_df.groupby('category')['price'].mean().sort_index()
    pandas_time1 = time.time() - start_time
    print(f"{pandas_time1:.4f} seconds")

    # SPARK
    print("Spark:  ", end="", flush=True)
    start_time = time.time()
    spark_result1 = spark_df.groupBy('category').agg(avg('price').alias('avg_price')).orderBy('category').collect()
    spark_time1 = time.time() - start_time
    print(f"{spark_time1:.4f} seconds")

    # Verify same results
    spark_values = {row['category']: round(row['avg_price'], 2) for row in spark_result1}
    pandas_values = pandas_result1.round(2).to_dict()
    print(f"✓ Results match: {pandas_values == spark_values}")

    # ============================================
    # TEST 2: Complex Aggregation (Multiple metrics)
    # ============================================
    print(f"\n📊 TEST 2: Multiple aggregations by region")
    print("-" * 50)

    # PANDAS
    print("Pandas: ", end="", flush=True)
    start_time = time.time()
    pandas_result2 = pandas_df.groupby('region').agg({
        'price': ['mean', 'std'],
        'quantity': 'sum',
        'user_id': 'count'
    })
    pandas_time2 = time.time() - start_time
    print(f"{pandas_time2:.4f} seconds")

    # SPARK
    print("Spark:  ", end="", flush=True)
    start_time = time.time()
    spark_result2 = spark_df.groupBy('region').agg(
        avg('price').alias('avg_price'),
        stddev('price').alias('std_price'),
        spark_sum('quantity').alias('total_quantity'),
        count('user_id').alias('user_count')
    ).collect()
    spark_time2 = time.time() - start_time
    print(f"{spark_time2:.4f} seconds")

    # ============================================
    # TEST 3: Filter + Aggregation
    # ============================================
    print(f"\n📊 TEST 3: Filter (price > 200) then aggregate")
    print("-" * 50)

    # PANDAS
    print("Pandas: ", end="", flush=True)
    start_time = time.time()
    filtered_pandas = pandas_df[pandas_df['price'] > 200]
    pandas_result3 = filtered_pandas.groupby('category')['quantity'].sum()
    pandas_time3 = time.time() - start_time
    print(f"{pandas_time3:.4f} seconds")

    # SPARK
    print("Spark:  ", end="", flush=True)
    start_time = time.time()
    spark_result3 = spark_df.filter(col('price') > 200) \
        .groupBy('category') \
        .agg(spark_sum('quantity').alias('total_quantity')) \
        .collect()
    spark_time3 = time.time() - start_time
    print(f"{spark_time3:.4f} seconds")

    # ============================================
    # MEMORY USAGE
    # ============================================
    print(f"\n💾 MEMORY USAGE")
    print("-" * 50)

    # Pandas memory
    pandas_memory = pandas_df.memory_usage(deep=True).sum() / 1024**2

    # Spark memory (approximate - Spark stores serialized)
    spark_df.cache()
    spark_df.count()  # Force cache
    spark_memory = spark_df.storageLevel.useMemory

    print(f"Pandas: {pandas_memory:.1f} MB")
    print(f"Spark:  Data distributed across partitions (not all in driver memory)")

    # Store results
    results.append({
        'size': size,
        'pandas_time1': pandas_time1,
        'spark_time1': spark_time1,
        'pandas_time2': pandas_time2,
        'spark_time2': spark_time2,
        'pandas_time3': pandas_time3,
        'spark_time3': spark_time3,
        'pandas_memory': pandas_memory,
        'speedup_agg1': pandas_time1/spark_time1 if spark_time1 > 0 else 0,
        'speedup_agg2': pandas_time2/spark_time2 if spark_time2 > 0 else 0
    })

    # Clear cache for next iteration
    spark_df.unpersist()

# ============================================
# SUMMARY TABLE
# ============================================
print("\n" + "="*80)
print("PERFORMANCE COMPARISON SUMMARY")
print("="*80)

summary_df = pd.DataFrame(results)
print("\n┌─────────────┬──────────────┬──────────────┬──────────────┬─────────────┐")
print("│ Rows        │ Pandas Avg   │ Spark Avg    │ Speedup      │ Memory      │")
print("│             │ (seconds)    │ (seconds)    │ (Spark faster)│ (Pandas MB) │")
print("├─────────────┼──────────────┼──────────────┼──────────────┼─────────────┤")

for _, row in summary_df.iterrows():
    avg_pandas = (row['pandas_time1'] + row['pandas_time2'] + row['pandas_time3']) / 3
    avg_spark = (row['spark_time1'] + row['spark_time2'] + row['spark_time3']) / 3
    speedup = avg_pandas / avg_spark if avg_spark > 0 else 0

    print(f"│ {row['size']:>9,} │ {avg_pandas:>12.4f} │ {avg_spark:>12.4f} │ {speedup:>12.1f}x │ {row['pandas_memory']:>10.0f} │")

print("└─────────────┴──────────────┴──────────────┴──────────────┴─────────────┘")

# ============================================
# DEMO: WHY SPARK IS CRITICAL FOR BIG DATA
# ============================================
print("\n" + "="*80)
print("WHY SPARK BECOMES ESSENTIAL FOR LARGER DATASETS")
print("="*80)

# Try a larger dataset (if memory allows)
print("\n🔬 Testing with 10,000,000 rows (simulated)...")
print("-" * 50)

try:
    large_size = 10_000_000
    print(f"Creating {large_size:,} rows of data...")

    # Create the data (same for both)
    np.random.seed(42)
    large_data = {
        'user_id': np.random.randint(1, 100000, large_size),
        'value': np.random.randn(large_size),
        'category': np.random.choice(['A', 'B', 'C', 'D'], large_size)
    }

    # Try pandas first
    print("\n【PANDAS】Attempting to load and process...")
    pandas_large = pd.DataFrame(large_data)
    pandas_mem = pandas_large.memory_usage(deep=True).sum() / 1024**3
    print(f"  ✓ Loaded successfully")
    print(f"  💾 Memory usage: {pandas_mem:.2f} GB")

    start_time = time.time()
    result = pandas_large.groupby('category')['value'].mean()
    pandas_time = time.time() - start_time
    print(f"  ⏱️  Aggregation time: {pandas_time:.2f} seconds")
    print(f"  ⚠️  This dataset uses {pandas_mem:.2f} GB of RAM")

    # Spark approach
    print("\n【SPARK】Processing same data...")
    spark_large = spark.createDataFrame(pandas_large)  # Reusing same data

    # Clear memory to show Spark's advantage
    del pandas_large

    start_time = time.time()
    spark_result = spark_large.groupBy('category').agg(avg('value')).collect()
    spark_time = time.time() - start_time
    print(f"  ⏱️  Aggregation time: {spark_time:.2f} seconds")
    print(f"  🚀 Spark processed without keeping all data in memory")

except MemoryError as e:
    print(f"\n  ✗ PANDAS FAILED: {e}")
    print("  ✓ SPARK would still work by distributing across disk/network")
    print("  💡 This is why Spark is important - it handles data larger than RAM!")

except Exception as e:
    print(f"\n  ✗ Error: {e}")
    print("  💡 Spark's strength is handling datasets that don't fit in memory!")

# ============================================
# REAL-WORLD SCENARIO
# ============================================
print("\n" + "="*80)
print("REAL-WORLD SCENARIO: When MUST you use Spark?")
print("="*80)

scenarios = """
┌─────────────────────────────────────────────────────────────────┐
│ SCENARIO                    │ PANDAS        │ SPARK            │
├─────────────────────────────┼───────────────┼──────────────────┤
│ 1GB of log files            │ ✅ Possible   │ ✅ Overkill      │
│ 50GB of log files           │ ❌ Impossible │ ✅ Perfect       │
│ Real-time streaming         │ ❌ No         │ ✅ Structured    │
│                             │               │    Streaming     │
│ Machine Learning on 100GB   │ ❌ No         │ ✅ MLlib         │
│ Processing 1TB of sensor data│ ❌ Crashes   │ ✅ Scales to PB  │
│ Graph analysis (social net) │ ❌ Too slow   │ ✅ GraphX        │
│ Running on cloud cluster    │ ❌ Single node│ ✅ Distributed   │
└─────────────────────────────────────────────────────────────────┘

KEY TAKEAWAYS:
• For < 1GB data: Pandas is simpler and often faster
• For 1-10GB data: Both work, but Spark scales better
• For > 10GB data: Spark is essential (pandas hits memory limits)
• For distributed computing: Spark is the clear choice
• For fault tolerance: Spark provides automatic recovery
"""

print(scenarios)

# Cleanup
spark.stop()
print("\n✓ Benchmark complete - Spark session closed")

FAIR COMPARISON: PANDAS vs SPARK on IDENTICAL DATA

📦 STEP 1: Creating identical dataset...

TESTING WITH 100,000 ROWS

📊 TEST 1: Average price by category
--------------------------------------------------
Pandas: 0.0211 seconds
Spark:  3.4407 seconds
✓ Results match: True

📊 TEST 2: Multiple aggregations by region
--------------------------------------------------
Pandas: 0.0423 seconds
Spark:  1.3750 seconds

📊 TEST 3: Filter (price > 200) then aggregate
--------------------------------------------------
Pandas: 0.0185 seconds
Spark:  1.6321 seconds

💾 MEMORY USAGE
--------------------------------------------------
Pandas: 13.5 MB
Spark:  Data distributed across partitions (not all in driver memory)

TESTING WITH 500,000 ROWS

📊 TEST 1: Average price by category
--------------------------------------------------
Pandas: 0.0666 seconds
Spark:  2.7546 seconds
✓ Results match: True

📊 TEST 2: Multiple aggregations by region
--------------------------------------------------
Pandas: 0.0